# **Setup & Components**

In [1]:
%pip install langchain langchain-core "langchain[openai]" langchain-text-splitters langchain-community bs4 langchain-huggingface sentence-transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 32.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 28.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.7/87.7 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.32.5 which is incompatible.


## LangSmith

In [2]:
import os
from google.colab import userdata

os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_API_KEY"] =  userdata.get("OPENROUTER_API_KEY")

#### 1. Select a chat model

In [3]:
# We use OpenRouter for the agent — add OPENROUTER_API_KEY to Colab Secrets (key icon in left sidebar)
# Get your key at https://openrouter.ai/keys
os.environ["OPENROUTER_API_KEY"] = userdata.get("OPENROUTER_API_KEY")

In [4]:
from langchain_openai import ChatOpenAI

# https://openrouter.ai/nvidia/nemotron-3-nano-30b-a3b:free
model_nemotron3_nano = ChatOpenAI(
    model="nvidia/nemotron-3-nano-30b-a3b:free",
    temperature=0,
    # OpenRouter instead of the default OpenAI API
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ.get("OPENROUTER_API_KEY"),
)

#### 2. Select an embeddings model

In [ ]:
# from langchain_openai import OpenAIEmbeddings
# embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2"
)

#### 3. Select a vector store

In [6]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

# **Indexing**


### Loading documents


In [7]:
from langchain_community.document_loaders import WebBaseLoader

# Load the Hugging Face technical blog
url = "https://huggingface.co/blog/rlhf"
loader = WebBaseLoader(web_paths=(url,))
docs = loader.load()

assert len(docs) == 1
print(f"Total characters: {len(docs[0].page_content)}")

Total characters: 24174


In [8]:
print(docs[0].page_content[:500])


























  
Illustrating Reinforcement Learning from Human Feedback (RLHF)






   Hugging Face        Models   Datasets   Spaces   Buckets new  Docs   Enterprise   Pricing       Log In Sign Up    Back to Articles 




		Illustrating Reinforcement Learning from Human Feedback (RLHF)
	
  Published December 9, 2022 Update on GitHub    Upvote 403               +397       Nathan Lambert  natolambert    Follow         Louis Castricato  LouisCastricato    Follow   guest      Leandro v


### Splitting documents


In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Split the document
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,
)
all_splits = text_splitter.split_documents(docs)

print(f"Split blog post into {len(all_splits)} sub-documents.")

Split blog post into 40 sub-documents.


### Storing documents


In [10]:
# Store in the vector database
document_ids = vector_store.add_documents(documents=all_splits)

print(f"Successfully indexed {len(all_splits)} sub-documents.")

Successfully indexed 40 sub-documents.


In [11]:
print(document_ids[:3])

['4f2c542b-1397-4a35-9149-de8ff01fd610', '9ccf9f3c-ee0e-4197-a816-85e258313964', 'd92cf12e-c6f3-440e-9a7d-8dd3f620d9e7']


# **Define the Tool and Create the Agent**

In [12]:
from typing import Literal
from langchain.tools import tool
from langchain.agents import create_agent
from langchain.messages import HumanMessage

@tool(response_format="content_and_artifact")
def retrieve_hf_blog_context(
    query: str,
    section: Literal["beginning", "middle", "end"]
):
    """Retrieve information from the Hugging Face RLHF blog post to help answer a query."""
    retrieved_docs = vector_store.similarity_search(query, k=2)
    serialized = "\n\n".join(
        (f"Source: {doc.metadata}\n"
        f"Content: {doc.page_content}")
        for doc in retrieved_docs
    )
    content, artifact = serialized, retrieved_docs
    return content, artifact

In [13]:
# Construct the Agent
agent = create_agent(
    model=model_nemotron3_nano,
    tools=[retrieve_hf_blog_context],
    system_prompt=(
        "You have access to a tool that retrieves context from a technical blog post about RLHF. "
        "Use the tool to help answer user queries."
    )
)

# **Multi-Step Execution**

In [14]:
# The multi-step prompt
query = (
    "What is the idea behind Reinforcement Learning from Human Feedback (RLHF)?\n\n"
    "Once you have defined it, look up what a 'reward model' is in this process."
)

# Invoke the agent
result = agent.invoke({"messages": [HumanMessage(query)]})

In [15]:
# Print the step-by-step trace
for msg in result["messages"]:
    msg.pretty_print()

================================ Human Message =================================

What is the idea behind Reinforcement Learning from Human Feedback (RLHF)?

Once you have defined it, look up what a 'reward model' is in this process.
================================== Ai Message ==================================
Tool Calls:
  retrieve_hf_blog_context (call_481691b51e394b21ac760957)
 Call ID: call_481691b51e394b21ac760957
  Args:
    section: beginning
    query: Reinforcement Learning from Human Feedback
================================= Tool Message =================================
Name: retrieve_hf_blog_context

Source: {'source': 'https://huggingface.co/blog/rlhf', 'title': 'Illustrating Reinforcement Learning from Human Feedback (RLHF)', 'description': 'We’re on a journey to advance and democratize artificial intelligence through open source and open science.', 'language': 'No language found.', 'start_index': 17758}
Content: TAMER: Training an Agent Manually via Evaluative Reinfo